# Coupled benchmark diagnostic code

In [6]:
"""
diagnose_integration_outputs.py

For each integration method, identifies exactly where the corrected
integration output is stored — expression matrix, embedding, or graph —
and reports the key names, shapes, and value ranges needed to design
the coupled benchmark geometry routing.

Usage:
    python diagnose_integration_outputs.py
"""

import anndata as ad
import numpy as np
from pathlib import Path

# ─────────────────────────────────────────────────────────────
# Update paths to match your actual output locations
# ─────────────────────────────────────────────────────────────
BASE = Path("/data1/esraa/Thesis-Project/Results/Integration/full_hvg_seed0/runs")

METHOD_PATHS = {
    "scvi":      BASE / "scvi_full_hvg_seed0/adata_scvi.h5ad",
    "scanvi":    BASE / "scanvi_full_hvg_seed0/adata_scanvi.h5ad",
    "scgen":     BASE / "scgen_full_hvg_seed0/adata_scgen.h5ad",
    "harmony":   BASE / "harmony_full_hvg_seed0/adata_harmony.h5ad",
    "fastmnn":   BASE / "fastmnn_full_hvg_seed0/adata_fastmnn_fastmnn_full_hvg_seed0.h5ad",
    "combat":    BASE / "combat_full_hvg_seed0/adata_combat.h5ad",
    "bbknn":     BASE / "bbknn_full_hvg_seed0/adata_bbknn.h5ad",
    "scanorama": BASE / "scanorama_full_hvg_seed0/adata_scanorama.h5ad",
    "liger":     BASE / "liger_full_hvg_seed0/adata_liger_liger_full_hvg_seed0.h5ad",
    "seurat":    BASE / "seurat_full_hvg_seed0/adata_seurat_seurat_full_hvg_seed0.h5ad",
}

# Keys to exclude from the obsm diagnostic — not integration outputs
EXCLUDE_OBSM = {
    "broad_signature_scores",
    "gse124395_cluster_signature_scores",
    "subtype_signature_scores",
    "_scvi_extra_categorical_covs",
}

SEP = "=" * 70


def arr_stats(arr) -> str:
    a = np.asarray(arr, dtype=np.float32)
    return f"shape={a.shape}  range=({a.min():.3f}, {a.max():.3f})  dtype={a.dtype}"


def sparse_stats(mat) -> str:
    return f"shape={mat.shape}  nnz={mat.nnz}  dtype={mat.dtype}"


def diagnose(method: str, path: Path) -> None:
    print(f"\n{SEP}")
    print(f"METHOD : {method}")
    print(f"FILE   : {path}")

    if not path.exists():
        print("  ✗ FILE NOT FOUND")
        print(SEP)
        return

    adata = ad.read_h5ad(str(path))
    print(f"SHAPE  : {adata.n_obs} cells × {adata.n_vars} genes")

    # ── adata.X ──────────────────────────────────────────────────────────────
    print(f"\n── adata.X (corrected expression?) ──")
    if adata.X is not None:
        import scipy.sparse as sp
        if sp.issparse(adata.X):
            xmin = float(adata.X.data.min()) if adata.X.data.size > 0 else 0.0
            xmax = float(adata.X.data.max()) if adata.X.data.size > 0 else 0.0
            xmean = float(adata.X.mean())
        else:
            arr = np.asarray(adata.X, dtype=np.float32)
            xmin, xmax, xmean = float(arr.min()), float(arr.max()), float(arr.mean())
        print(f"  shape={adata.X.shape}  range=({xmin:.3f}, {xmax:.3f})  mean={xmean:.4f}")
        if xmax < 15:
            print(f"  → likely log-normalised (max < 15)")
        elif xmax < 200:
            print(f"  → possibly corrected expression (moderate range)")
        else:
            print(f"  → possibly raw counts or uncorrected (max = {xmax:.1f})")
    else:
        print("  X is None")

    # ── adata.layers ─────────────────────────────────────────────────────────
    print(f"\n── adata.layers ──")
    if adata.layers:
        for k in adata.layers.keys():
            import scipy.sparse as sp
            X = adata.layers[k]
            if sp.issparse(X):
                mn = float(X.data.min()) if X.data.size > 0 else 0.0
                mx = float(X.data.max()) if X.data.size > 0 else 0.0
            else:
                a = np.asarray(X, dtype=np.float32)
                mn, mx = float(a.min()), float(a.max())
            print(f"  {k:30s}  shape={X.shape}  range=({mn:.3f}, {mx:.3f})")
    else:
        print("  (no layers)")

    # ── adata.obsm — integration outputs ─────────────────────────────────────
    print(f"\n── adata.obsm (integration outputs) ──")
    integration_keys = []
    for k, v in adata.obsm.items():
        if k in EXCLUDE_OBSM:
            continue
        arr = np.asarray(v, dtype=np.float32)
        stats = arr_stats(arr)
        flag = ""
        if k == "X_emb":
            flag = "  ← X_emb (standardised alias)"
        elif k.startswith("X_pca") and "harmony" not in k and "combat" not in k:
            flag = "  ← uncorrected PCA (ignore)"
        elif k.startswith("X_pca"):
            flag = "  ← corrected PCA"
        elif k.startswith("X_umap"):
            flag = "  ← UMAP (visualisation only)"
        elif k.startswith("X_diffmap"):
            flag = "  ← diffmap embedding"
        print(f"  {k:45s}  {stats}{flag}")
        if flag not in ("  ← uncorrected PCA (ignore)", "  ← UMAP (visualisation only)"):
            integration_keys.append(k)

    # ── adata.obsp — graph outputs ────────────────────────────────────────────
    print(f"\n── adata.obsp (graph outputs) ──")
    graph_conn_keys = []
    for k, v in adata.obsp.items():
        print(f"  {k:50s}  {sparse_stats(v)}")
        if "connectivities" in k:
            graph_conn_keys.append(k)

    # ── adata.uns — neighbors metadata ───────────────────────────────────────
    print(f"\n── adata.uns (neighbors-related keys) ──")
    neighbors_uns_keys = []
    for k in adata.uns.keys():
        if "neighbor" in k.lower() or "bbknn" in k.lower() or "diffmap" in k.lower():
            print(f"  {k}")
            if "neighbor" in k.lower():
                neighbors_uns_keys.append(k)

    # ── ROUTING VERDICT ───────────────────────────────────────────────────────
    print(f"\n── Routing Verdict ──")

    has_graph = len(graph_conn_keys) > 0
    has_emb   = "X_emb" in adata.obsm

    # Check if X is corrected expression (ComBat/MNN pattern)
    # These methods have no X_emb and no graph keys that are corrected
    # Their corrected output IS adata.X
    method_specific_embedding = [
        k for k in adata.obsm.keys()
        if k not in EXCLUDE_OBSM
        and not k.startswith("X_pca")
        and not k.startswith("X_umap")
        and not k.startswith("X_diffmap")
        and k != "X_emb"
    ]

    if has_graph and not has_emb and method == "bbknn":
        print(f"  ROUTE C — Graph-native")
        print(f"  Corrected graph keys:")
        for k in graph_conn_keys:
            print(f"    obsp['{k}']")
        for k in neighbors_uns_keys:
            print(f"    uns['{k}']")

    elif has_emb:
        emb = np.asarray(adata.obsm["X_emb"])
        print(f"  ROUTE B — Embedding")
        print(f"  Primary key : obsm['X_emb']  shape={emb.shape}")
        if method_specific_embedding:
            print(f"  Native key  : obsm['{method_specific_embedding[0]}']")

    else:
        print(f"  ROUTE A — Expression-corrected (use adata.X for PCA)")
        print(f"  X range: ({xmin:.3f}, {xmax:.3f})")

    print(SEP)


# ─────────────────────────────────────────────────────────────
# Run
# ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    for method, path in METHOD_PATHS.items():
        diagnose(method, path)

    print("\n")
    print("=" * 70)
    print("ROUTING SUMMARY")
    print("=" * 70)
    print("""
Route A — Full pipeline (PCA → graph → UMAP from corrected adata.X)
  Methods: combat, mnn

Route B — Embedding pipeline (skip PCA, graph → UMAP from X_emb)
  Methods: scvi, scanvi, scgen, harmony, fastmnn, scanorama, liger, seurat

Route C — Graph pipeline (graph already exists, UMAP only)
  Methods: bbknn
""")


METHOD : scvi
FILE   : /data1/esraa/Thesis-Project/Results/Integration/full_hvg_seed0/runs/scvi_full_hvg_seed0/adata_scvi.h5ad


SHAPE  : 299891 cells × 4000 genes

── adata.X (corrected expression?) ──
  shape=(299891, 4000)  range=(0.112, 6275.803)  mean=0.1480
  → possibly raw counts or uncorrected (max = 6275.8)

── adata.layers ──
  counts                          shape=(299891, 4000)  range=(1.000, 20968.000)
  log1p_norm                      shape=(299891, 4000)  range=(0.288, 8.614)

── adata.obsm (integration outputs) ──
  X_emb                                          shape=(299891, 50)  range=(-5.304, 8.033)  dtype=float32  ← X_emb (standardised alias)
  X_pca                                          shape=(299891, 50)  range=(-45.954, 76.091)  dtype=float32  ← uncorrected PCA (ignore)
  X_scvi_full_hvg_seed0                          shape=(299891, 50)  range=(-5.304, 8.033)  dtype=float32
  X_umap_scvi_full_hvg_seed0                     shape=(299891, 2)  range=(-10.000, 10.000)  dtype=float32  ← UMAP (visualisation only)

── adata.obsp (graph outputs) ──
  neighbors_scvi_full_hvg_seed0_connectivitie

In [4]:
import anndata as ad

adata = ad.read_h5ad("/data1/esraa/Thesis-Project/Results/Integration/full_hvg_seed0/runs/bbknn_full_hvg_seed0/adata_bbknn.h5ad")
print("obsp keys:", list(adata.obsp.keys()))
print("uns keys:", [k for k in adata.uns.keys() if "neighbor" in k.lower() or "bbknn" in k.lower()])

obsp keys: ['bbknn_full_hvg_seed0_connectivities', 'bbknn_full_hvg_seed0_distances']
uns keys: ['diffmap_evals_X_diffmap_bbknn_full_hvg_seed0', 'leiden_bbknn_full_hvg_seed0', 'neighbors_bbknn_full_hvg_seed0', 'umap_X_umap_bbknn_full_hvg_seed0']


In [3]:
import anndata as ad
from pathlib import Path

BASE = Path("/data1/esraa/Thesis-Project/Results/Integration/full_hvg_seed0/runs")

files = {
    "scvi":      BASE / "scvi_full_hvg_seed0/adata_scvi.h5ad",
    "scanvi":    BASE / "scanvi_full_hvg_seed0/adata_scanvi.h5ad",
    "scgen":     BASE / "scgen_full_hvg_seed0/adata_scgen.h5ad",
    "harmony":   BASE / "harmony_full_hvg_seed0/adata_harmony.h5ad",
    "fastmnn":   BASE / "fastmnn_full_hvg_seed0/adata_fastmnn_fastmnn_full_hvg_seed0.h5ad",
    "combat":    BASE / "combat_full_hvg_seed0/adata_combat.h5ad",
    "bbknn":     BASE / "bbknn_full_hvg_seed0/adata_bbknn.h5ad",
    "scanorama": BASE / "scanorama_full_hvg_seed0/adata_scanorama.h5ad",
    "liger":     BASE / "liger_full_hvg_seed0/adata_liger_liger_full_hvg_seed0.h5ad",
    "seurat":    BASE / "seurat_full_hvg_seed0/adata_seurat_seurat_full_hvg_seed0.h5ad",
}

print(f"{'Method':<12} {'X_emb shape':<20} {'Expected 50?'}")
print("-" * 45)
for method, path in files.items():
    adata = ad.read_h5ad(str(path))
    emb = adata.obsm.get("X_emb")
    if emb is not None:
        import numpy as np
        shape = np.asarray(emb).shape
        ok = "✓" if shape[1] == 50 else f"✗ got {shape[1]}"
        print(f"{method:<12} {str(shape):<20} {ok}")
    else:
        print(f"{method:<12} {'MISSING':<20} ✗")

Method       X_emb shape          Expected 50?
---------------------------------------------


scvi         (299891, 50)         ✓
scanvi       (299891, 50)         ✓
scgen        (299891, 30)         ✗ got 30
harmony      (299891, 50)         ✓
fastmnn      (299891, 50)         ✓
combat       (299891, 50)         ✓
bbknn        (299891, 50)         ✓
scanorama    (299891, 50)         ✓
liger        (291545, 50)         ✓
seurat       (299891, 30)         ✗ got 30


In [6]:
import anndata as ad
import os

base = "/data1/esraa/Thesis-Project/Results/Integration/full_hvg_seed0/runs"
methods = ["scvi", "scanvi", "scgen", "harmony", "fastmnn", "mnn", 
           "combat", "bbknn", "scanorama", "liger", "seurat"]

for method in methods:
    path = f"/data1/esraa/Thesis-Project/Results/Integration/full_hvg_seed0/runs/{method}_full_hvg_seed0/adata_{method}.h5ad"
    if not os.path.exists(path):
        print(f"{method:12s}  FILE NOT FOUND")
        continue
    adata = ad.read_h5ad(path)
    layers = list(adata.layers.keys())
    print(f"{method:12s}  layers: {layers}")


scvi          layers: ['counts', 'log1p_norm']
scanvi        layers: ['counts', 'log1p_norm']
scgen         layers: ['counts', 'log1p_norm']
harmony       layers: ['counts', 'log1p_norm']
fastmnn       layers: ['counts', 'log1p_norm']
mnn           FILE NOT FOUND
combat        layers: ['counts', 'log1p_norm']
bbknn         layers: ['counts', 'log1p_norm']
scanorama     layers: []
liger         layers: ['counts', 'log1p_norm']
seurat        layers: ['counts', 'log1p_norm']


In [7]:
import sys
from pathlib import Path
# install rpy2 scib for reducability
PROJECT_ROOT = Path.cwd().parent 
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("SRC added:", SRC)

#re-run at the beginning of each experiment 
from thesis_project.Integration.Integration_benchmark.seeds import set_global_seed
SEED = 0
set_global_seed(SEED, use_torch=True)
# scib
import importlib
import thesis_project.Integration.Integration_benchmark.scib_compat as scib_compat
importlib.reload(scib_compat)
print("scib_compat loaded from:", scib_compat.__file__)


SRC added: /data1/esraa/Thesis-Project/src


/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


scib_compat loaded from: /data1/esraa/Thesis-Project/src/thesis_project/Integration/Integration_benchmark/scib_compat.py


In [ ]:
!pip install scib==1.1.7 scikit-learn umap-learn
# Install kBET dependencies (requires R and rpy2)
# conda install -c conda-forge mnnpy
!pip install rpy2 anndata2ri tzlocal

# construct the 11 adata objects to perform trajectory inference

In [9]:
import os
import scanpy as sc

base_dir = "/data1/esraa/Thesis-Project/Results/Integration/full_hvg_seed0/runs"

methods = [
    "bbknn_full_hvg_seed0", "combat_full_hvg_seed0", "fastmnn_full_hvg_seed0", "harmony_full_hvg_seed0",
    "liger_full_hvg_seed0", "mnn_full_hvg_seed0", "scanorama_full_hvg_seed0", "scanvi_full_hvg_seed0",
    "scgen_full_hvg_seed0", "scvi_full_hvg_seed0", "seurat_full_hvg_seed0"
]

all_integrated_adata = {}

for method in methods:
    short_name = method.replace("_full_hvg_seed0", "")
    filename = f"adata_{short_name}.h5ad"
    path = os.path.join(base_dir, method, filename)

    if os.path.exists(path):
        print(f"Loading {path}")
        all_integrated_adata[short_name] = sc.read_h5ad(path)
    else:
        print(f"Missing: {path}")
        
print("Loaded integrated adatas for methods:", list(all_integrated_adata.keys()))

Loading /data1/esraa/Thesis-Project/Results/Integration/full_hvg_seed0/runs/bbknn_full_hvg_seed0/adata_bbknn.h5ad


Loading /data1/esraa/Thesis-Project/Results/Integration/full_hvg_seed0/runs/combat_full_hvg_seed0/adata_combat.h5ad
Loading /data1/esraa/Thesis-Project/Results/Integration/full_hvg_seed0/runs/fastmnn_full_hvg_seed0/adata_fastmnn.h5ad
Loading /data1/esraa/Thesis-Project/Results/Integration/full_hvg_seed0/runs/harmony_full_hvg_seed0/adata_harmony.h5ad
Loading /data1/esraa/Thesis-Project/Results/Integration/full_hvg_seed0/runs/liger_full_hvg_seed0/adata_liger.h5ad
Missing: /data1/esraa/Thesis-Project/Results/Integration/full_hvg_seed0/runs/mnn_full_hvg_seed0/adata_mnn.h5ad
Loading /data1/esraa/Thesis-Project/Results/Integration/full_hvg_seed0/runs/scanorama_full_hvg_seed0/adata_scanorama.h5ad
Loading /data1/esraa/Thesis-Project/Results/Integration/full_hvg_seed0/runs/scanvi_full_hvg_seed0/adata_scanvi.h5ad
Loading /data1/esraa/Thesis-Project/Results/Integration/full_hvg_seed0/runs/scgen_full_hvg_seed0/adata_scgen.h5ad
Loading /data1/esraa/Thesis-Project/Results/Integration/full_hvg_seed0/

In [ ]:
for k in all_integrated_adata:
    print(k,"    ", all_integrated_adata[k].obs["cell_subtype_L2"].value_counts())  
    print("====================") 

bbknn      cell_subtype_L2
NK cells                     13308
Pro-metastatic Hepatocyte     9195
CD8 T cells                   8669
CD8 T (GZMK+)                 8212
CD4 T cells                   8204
                             ...  
Lymphatic Endothelial           19
Inflamed Cholangiocyte          17
Pericyte/Stellate/vSMC           8
IGF2/H19+ Fibroblast             7
ACKR1+ Endothelial               3
Name: count, Length: 115, dtype: int64
combat      cell_subtype_L2
NK cells                     13308
Pro-metastatic Hepatocyte     9195
CD8 T cells                   8669
CD8 T (GZMK+)                 8212
CD4 T cells                   8204
                             ...  
Lymphatic Endothelial           19
Inflamed Cholangiocyte          17
Pericyte/Stellate/vSMC           8
IGF2/H19+ Fibroblast             7
ACKR1+ Endothelial               3
Name: count, Length: 115, dtype: int64
fastmnn      cell_subtype_L2
NK cells                     13308
Pro-metastatic Hepatocyte     919

In [15]:
import pandas as pd

pd.set_option("display.max_rows", None)
df = (
    all_integrated_adata["bbknn"]
    .obs["cell_subtype_L2"]
    .value_counts()
    .rename_axis("cell_subtype_L2")
    .reset_index(name="count")
)
print(df)


                  cell_subtype_L2  count
0                        NK cells  13308
1       Pro-metastatic Hepatocyte   9195
2                     CD8 T cells   8669
3                   CD8 T (GZMK+)   8212
4                     CD4 T cells   8204
5                     Endothelial   7921
6                         B cells   6961
7                      MAIT cells   6391
8                      Hepatocyte   6377
9                            cDC2   6196
10     Pro-tumorigenic Hepatocyte   6020
11                   Plasma cells   5583
12                  CD4 T (TCF7+)   5280
13                  Cholangiocyte   4316
14              MARCO+ Macrophage   3961
15                   Regulatory T   3957
16                CD8 T (CX3CR1+)   3928
17                      cNK cells   3923
18                           SAMs   3919
19                 CD8 T (KLRD1+)   3747
20                    Cytotoxic T   3703
21              CD4/CD8 T (CCR7+)   3418
22                  Kupffer cells   3212
23    Monocyte-d

# HVG Data loading 


In [1]:
import anndata as ad

# Load the HVG subset
hvg_path = "/data1/esraa/Thesis-Project/Data/Processed_data/post_HVG_intersection/concatenated_hvg.h5ad"
adata_all_hvg = ad.read_h5ad(hvg_path)

print(f"Loaded HVG data:")
print(f"  Shape: {adata_all_hvg.n_obs:,} cells × {adata_all_hvg.n_vars:,} genes")
print(f"  Batch key: {adata_all_hvg.obs['dataset'].unique()}")


Loaded HVG data:
  Shape: 299,891 cells × 4,000 genes
  Batch key: ['GSE149614', 'GSE151530', 'GSE125449', 'GSE115469', 'GSE140228', 'GSE146409', 'GSE138709', 'GSE136103', 'GSE124395']
Categories (9, object): ['GSE115469', 'GSE124395', 'GSE125449', 'GSE136103', ..., 'GSE140228', 'GSE146409', 'GSE149614', 'GSE151530']


In [11]:
import pandas as pd
from pathlib import Path

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    df_major_celltypes = (
        adata_all_hvg.obs["cell_subtype_L2"]
        .value_counts()
        .rename_axis("cell_subtype_L2")
        .reset_index(name="count")
    )
    print(df_major_celltypes)

    series_counts = adata_all_hvg.obs["cell_subtype_L2"].value_counts()
    print(series_counts)

# Save results to Excel
output_path = Path("/data1/esraa/Thesis-Project/Results/cell_subtype_L2_counts.xlsx")
output_path.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(output_path) as writer:
    df_major_celltypes.to_excel(writer, sheet_name="counts_table", index=False)
    (
        series_counts.rename_axis("cell_subtype_L2")
        .reset_index(name="count")
        .to_excel(writer, sheet_name="value_counts", index=False)
    )

print(f"Saved Excel file to: {output_path}")

                  cell_subtype_L2  count
0                        NK cells  13308
1       Pro-metastatic Hepatocyte   9195
2                     CD8 T cells   8669
3                   CD8 T (GZMK+)   8212
4                     CD4 T cells   8204
5                     Endothelial   7921
6                         B cells   6961
7                      MAIT cells   6391
8                      Hepatocyte   6377
9                            cDC2   6196
10     Pro-tumorigenic Hepatocyte   6020
11                   Plasma cells   5583
12                  CD4 T (TCF7+)   5280
13                  Cholangiocyte   4316
14              MARCO+ Macrophage   3961
15                   Regulatory T   3957
16                CD8 T (CX3CR1+)   3928
17                      cNK cells   3923
18                           SAMs   3919
19                 CD8 T (KLRD1+)   3747
20                    Cytotoxic T   3703
21              CD4/CD8 T (CCR7+)   3418
22                  Kupffer cells   3212
23    Monocyte-d

# Smoke test 

In [7]:
import numpy as np

def smoke_subset(adata, n=20000, seed=0, stratify_by="dataset"):
    if adata.n_obs <= n:
        return adata.copy()

    rng = np.random.default_rng(seed)

    if stratify_by is None or stratify_by not in adata.obs:
        idx = rng.choice(adata.n_obs, size=n, replace=False)
        return adata[idx].copy()

    groups = adata.obs[stratify_by].astype(str).values
    uniq, counts = np.unique(groups, return_counts=True)
    props = counts / counts.sum()
    take_per = np.maximum(1, (props * n).astype(int))

    picked = []
    for u, k in zip(uniq, take_per):
        idx_u = np.where(groups == u)[0]
        k = min(k, idx_u.size)
        picked.append(rng.choice(idx_u, size=k, replace=False))

    idx = np.unique(np.concatenate(picked))
    if idx.size > n:
        idx = rng.choice(idx, size=n, replace=False)

    return adata[idx].copy()
ad_smoke = smoke_subset(adata_all_hvg, n=20000, seed=0, stratify_by="dataset")
print(ad_smoke.shape)
print(ad_smoke.obs["dataset"].value_counts().head())
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent  # Thesis-Project/
SMOKE_DIR = PROJECT_ROOT / "Results" / "Smoke"
SMOKE_DIR.mkdir(parents=True, exist_ok=True)



(19997, 4000)
dataset
GSE149614    4446
GSE140228    4221
GSE136103    3918
GSE151530    3215
GSE138709    1948
Name: count, dtype: int64


In [8]:
# Save the smoke test AnnData to /data1/esraa/Thesis-Project/Data/smoke_data/smoke_test.h5ad
ad_smoke.write_h5ad("/data1/esraa/Thesis-Project/Data/smoke_data/smoke_test.h5ad")
print("Saved smoke test data to /data1/esraa/Thesis-Project/Data/smoke_data/smoke_test.h5ad")

Saved smoke test data to /data1/esraa/Thesis-Project/Data/smoke_data/smoke_test.h5ad


# Start from here 

In [1]:
import sys
from pathlib import Path
# install rpy2 scib for reducability
PROJECT_ROOT = Path.cwd().parent 
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("SRC added:", SRC)

#re-run at the beginning of each experiment 
from thesis_project.Integration.Integration_benchmark.seeds import set_global_seed
SEED = 0
set_global_seed(SEED, use_torch=True)
# scib
import importlib
import thesis_project.Integration.Integration_benchmark.scib_compat as scib_compat
importlib.reload(scib_compat)
print("scib_compat loaded from:", scib_compat.__file__)

# read the smoke test data from /data1/esraa/Thesis-Project/Data/smoke_data/smoke_test.h5ad
import anndata as ad
smoke_path = "/data1/esraa/Thesis-Project/Data/smoke_data/smoke_test.h5ad"
ad_smoke = ad.read_h5ad(smoke_path)
print(f"Loaded smoke test data:")
print(f"  Shape: {ad_smoke.n_obs:,} cells × {ad_smoke.n_vars:,} genes")
print(f"  Batch key: {ad_smoke.obs['dataset'].unique()}")


SRC added: /data1/esraa/Thesis-Project/src


/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


scib_compat loaded from: /data1/esraa/Thesis-Project/src/thesis_project/Integration/Integration_benchmark/scib_compat.py
Loaded smoke test data:
  Shape: 19,997 cells × 4,000 genes
  Batch key: ['GSE149614', 'GSE151530', 'GSE125449', 'GSE115469', 'GSE140228', 'GSE146409', 'GSE138709', 'GSE136103', 'GSE124395']
Categories (9, object): ['GSE115469', 'GSE124395', 'GSE125449', 'GSE136103', ..., 'GSE140228', 'GSE146409', 'GSE149614', 'GSE151530']


In [2]:
for cov in ["donor_id", "technology", "platform", "tissue", "tumor_status"]:
    if cov in ad_smoke.obs:
        col = ad_smoke.obs[cov].astype(str).replace("nan", "Unknown")
        n_missing = ad_smoke.obs[cov].isna().sum()
        n_unknown = (col == "Unknown").sum()
        pct_bad = 100 * (n_missing + n_unknown) / ad_smoke.n_obs
        n_unique = col.nunique()
        print(f"\n{'='*60}") 
        print(f"{cov}: {pct_bad:.1f}% missing/unknown | {n_unique} unique values")
        print(col.value_counts(dropna=False).to_string())
    else:
        print(f"\n{cov}: NOT PRESENT in ad_smoke.obs")


donor_id: 6.8% missing/unknown | 106 unique values
donor_id
D20171109         1339
D20171215         1237
ICC_18             832
HCC08              831
healthy2           721
Unknown            676
D20180108          671
D20180110          654
HCC06              568
healthy4           565
HCC10              533
HCC05              506
healthy3           483
ICC_23             479
HCC03              459
cirrhotic2         394
HCC04              394
HCC07              387
S002               364
ICC_24             360
healthy5           351
cirrhotic1         349
cirrhotic3         335
S009               330
D20180116          320
HCC09              319
cirrhotic4         311
S004               281
S011               268
HCC02              235
healthy1           233
S006               232
S003               224
HCC01              214
P3TLH              202
cirrhotic5         176
ICC_20             148
S001               142
S012               137
S013               131
ICC_25             

# Mnn

In [15]:
# Example usage snippet (Jupyter / .py)
from pathlib import Path
import importlib
from thesis_project.Integration.Integration_methods import mnn as mnn_mod
importlib.reload(mnn_mod)

PROJECT_ROOT = Path("/data1/esraa/Thesis-Project")
outdir = PROJECT_ROOT / "Results" / "Smoke" / "mnn_smoke"
outdir.mkdir(parents=True, exist_ok=True)

cfg = mnn_mod.MNNConfig(
    batch_key="dataset",
    label_key="major_celltype_l1",
    run_tag="mnn_smoke",
    seed=0,
    require_hvg=True,
    max_hvgs=1500,
    k=10,
    nn_method="hnsw",
    bp_workers=4,
    svd_dim=30,
    auto_merge=True,
    restrict_n_per_batch=1500,
    r_timeout_s=3600,
    save_h5ad=False,
)

ad_out, metrics, perf_df = mnn_mod.run_mnncorrect_via_r(
    adata_in=ad_smoke,
    outdir=str(outdir),
    cfg=cfg
)

print("Done. Metric keys:", list(metrics.keys()))
print(perf_df)




[Python] Starting MNN run: mnn_smoke
[Python] Input cells: 19997, genes: 4000


[Python] HVGs exported for mnnCorrect subset.row: 1500
[Python] Launching R subprocess at 00:16:45
[Python] Command: Rscript /data1/esraa/Thesis-Project/src/thesis_project/Integration/Integration_methods/R/run_mnncorrect.R /data1/esraa/Thesis-Project/Results/Smoke/mnn_smoke/adata_for_mnn_mnn_smoke.h5ad dataset /data1/esraa/Thesis-Project/Results/Smoke/mnn_smoke/mnn_smoke 10 /data1/esraa/Thesis-Project/Results/Smoke/mnn_smoke/mnn_genes_mnn_smoke.csv 50 0 0.1 TRUE TRUE 30 TRUE TRUE hnsw 4 1500
[R] === MNN START ===
[R] Start time: 2026-04-05 00:16:49.928478
[R] Seed: 0
[R] Reading H5AD: /data1/esraa/Thesis-Project/Results/Smoke/mnn_smoke/adata_for_mnn_mnn_smoke.h5ad


KeyboardInterrupt: 

# fastmnn


In [15]:
import importlib
from thesis_project.Integration.Integration_methods import fastmnn as fastmnn
from thesis_project.Integration.Integration_methods.fastmnn import run_fastmnn_via_r, FastMNNConfig
importlib.reload(fastmnn)
PROJECT_ROOT = Path("/data1/esraa/Thesis-Project")
outdir = PROJECT_ROOT / "Results" / "Smoke" / "fastmnn_smoke"

cfg = FastMNNConfig(
    batch_key="dataset",
    label_key="major_celltype_l1",
    run_tag="fastmnn_smoke",
    seed=0, 
    max_hvgs=4000,   
    d=50,           # smoke
    k=20,
    assay_type="X", # IMPORTANT for data (lognorm in AnnData .X)
    save_h5ad=True
)

ad_out, metrics, perf_df = run_fastmnn_via_r(
    adata_in=ad_smoke,      
    outdir=str(outdir),
    cfg=cfg
)

print("Done.")


/data1/esraa/Thesis-Project/src/thesis_project/Integration/Integration_benchmark/graph.py:226: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/isolated_labels.py:311: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  batch_per_lab = tmp.groupby(label_key).agg({batch_key: "count"})


Recompute neighbors on rep X_fastmnn_fastmnn_smoke instead of None
Cluster for iso_label_0.2 with leiden


/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/clustering.py:96: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  cluster_function(adata, resolution=res, key_added=resolution_key, **kwargs)


Cluster for iso_label_0.4 with leiden
Cluster for iso_label_0.6 with leiden
Cluster for iso_label_0.8 with leiden
Cluster for iso_label_1.0 with leiden
Cluster for iso_label_1.2 with leiden
Cluster for iso_label_1.4 with leiden
Cluster for iso_label_1.6 with leiden
Cluster for iso_label_1.8 with leiden
Cluster for iso_label_2.0 with leiden


/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/isolated_labels.py:311: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  batch_per_lab = tmp.groupby(label_key).agg({batch_key: "count"})
  0%|          | 0/9 [00:00<?, ?it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/preprocessing/_pca/__init__.py:226: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can b

...

Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 6
Adding diffusion to step 7
Adding diffusion to step 8


/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


Adding diffusion to step 4


/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/graph_connectivity.py:56: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(labels)
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/graph_connectivity.py:56: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(labels)
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/graph_connectivity.py:56: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(labels)
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/graph_connectivity.py:56: FutureWarning: pandas.value_counts is deprecated and will be removed in a

Done.


# Liger 
(raw counts → normalize → selectGenes → scaleNotCenter → iNMF → alignFactors → export H.norm)

In [3]:
from pathlib import Path
import importlib
from thesis_project.Integration.Integration_methods import liger as liger_mod
importlib.reload(liger_mod)

PROJECT_ROOT = Path("/data1/esraa/Thesis-Project")
outdir = PROJECT_ROOT / "Results" / "Smoke" / "liger_smoke"
outdir.mkdir(parents=True, exist_ok=True)

cfg = liger_mod.LigerConfig(
    batch_key="dataset",
    label_key="major_celltype_l1",
    run_tag="liger_smoke",
    seed=0,

    input_layer_raw="counts",       # Use adata.layers["counts"] for raw counts

    require_hvg=True,
    hvg_key="highly_variable",
    max_hvgs=None,                  # None => use ALL HVGs 

    # LIGER params
    k_factors=15,
    lambda_reg=5.0,
    n_iters=20,

    save_h5ad=False,
)

ad_out, metrics, perf_df = liger_mod.run(ad_smoke, str(outdir), cfg)

[Python] Using raw counts from layer 'counts'
[Python] Dropping 556 cells with zero total raw counts before LIGER
[Python] Example dropped cells: GSE115469_P1TLH_AAACCTGTCTAAGCCA_1, GSE115469_P1TLH_AACTCAGAGGCGTACA_1, GSE115469_P1TLH_AAGCCGCAGCTAGTGG_1, GSE115469_P1TLH_ACACCAATCCTCAATT_1, GSE115469_P1TLH_ACACCGGAGGATGGAA_1, GSE115469_P1TLH_ACAGCTAAGGAACTGC_1, GSE115469_P1TLH_ACCAGTATCTATCCTA_1, GSE115469_P1TLH_ACCGTAACACAACTGT_1, GSE115469_P1TLH_ACGGAGAAGCGCCTTG_1, GSE115469_P1TLH_AGAGCTTAGCCACGCT_1
[Python] Using raw counts from layer 'counts'
[Python] Raw-count validation passed (min=1.000, zero_count_cells=0)
[Python] Writing compact temporary H5AD for R: /data1/esraa/Thesis-Project/Results/Smoke/liger_smoke/adata_for_liger_liger_smoke.h5ad
[Python] Using all 4000 externally supplied HVGs
[Python] HVG list written: /data1/esraa/Thesis-Project/Results/Smoke/liger_smoke/liger_genes_liger_smoke.txt (n=4000)
[Python] Calling R LIGER with: Rscript --vanilla /data1/esraa/Thesis-Project/sr

/data1/esraa/Thesis-Project/src/thesis_project/Integration/Integration_benchmark/graph.py:226: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/isolated_labels.py:311: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  batch_per_lab = tmp.groupby(label_key).agg({batch_key: "count"})
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/clustering.py:96: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults pl

Cluster for iso_label_0.2 with leiden
Cluster for iso_label_0.4 with leiden
Cluster for iso_label_0.6 with leiden
Cluster for iso_label_0.8 with leiden
Cluster for iso_label_1.0 with leiden
Cluster for iso_label_1.2 with leiden
Cluster for iso_label_1.4 with leiden
Cluster for iso_label_1.6 with leiden
Cluster for iso_label_1.8 with leiden
Cluster for iso_label_2.0 with leiden
Cluster for iso_label_0.2 with leiden
Cluster for iso_label_0.4 with leiden
Cluster for iso_label_0.6 with leiden
Cluster for iso_label_0.8 with leiden
Cluster for iso_label_1.0 with leiden
Cluster for iso_label_1.2 with leiden
Cluster for iso_label_1.4 with leiden
Cluster for iso_label_1.6 with leiden
Cluster for iso_label_1.8 with leiden
Cluster for iso_label_2.0 with leiden


/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/isolated_labels.py:311: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  batch_per_lab = tmp.groupby(label_key).agg({batch_key: "count"})
  0%|          | 0/8 [00:00<?, ?it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/preprocessing/_pca/__init__.py:226: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can b

mean silhouette per group:                silhouette_score
group                          
B cells                0.857050
CAFs                   0.873998
Cholangiocyte          0.839606
Endothelial            0.910493
Fibroblast             0.921849
Hepatocyte             0.924374
Kupffer cells          0.763047
LSECs                  0.879528
Malignant              0.823727
Mesenchymal            0.827040
Myeloid                0.906143
NK cells               0.875827
Plasma cells           0.910206
T cells                0.859718
TAMs                   0.794023
TECs                   0.842604


...

/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)


Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 6
Adding diffusion to step 7
Adding diffusion to step 8
Adding diffusion to step 9
Adding diffusion to step 10
Adding diffusion to step 11
Adding diffusion to step 12
Adding diffusion to step 13


/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)


Adding diffusion to step 4
Adding diffusion to step 5


/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/graph_connectivity.py:56: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use

# ScGen

In [4]:
import importlib
import thesis_project.Integration.Integration_methods.scgen as scgen_mod
importlib.reload(scgen_mod)

from thesis_project.Integration.Integration_methods.scgen import run, ScgenConfig
    
cfg = ScgenConfig(
    batch_key="dataset",
    label_key="major_celltype_l1",
    run_tag="scgen_smoke",
    save_h5ad=False,
    # optional: align to one batch per label (if you want)
    # reference_batch="GSE149614",
)

outdir = PROJECT_ROOT / "Results" / "Smoke" / "scgen_smoke"

ad_out, metrics, perf = run(
    adata_in=ad_smoke,
    outdir=str(outdir),
    cfg=cfg,
) 


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA RTX 6000 Ada Generation') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Training:   0%|          | 0/200 [00:00<?, ?it/s]

Monitored metric elbo_validation did not improve in the last 25 records. Best score: 1129.355. Signaling Trainer to stop.
[scGen] Using direct encoder access to avoid sqrt(None) bug.
[scGen] Latent shape: (19997, 30)


/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/isolated_labels.py:311: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  batch_per_lab = tmp.groupby(label_key).agg({batch_key: "count"})


Cluster for iso_label_0.2 with leiden
Cluster for iso_label_0.4 with leiden
Cluster for iso_label_0.6 with leiden
Cluster for iso_label_0.8 with leiden
Cluster for iso_label_1.0 with leiden
Cluster for iso_label_1.2 with leiden
Cluster for iso_label_1.4 with leiden
Cluster for iso_label_1.6 with leiden
Cluster for iso_label_1.8 with leiden
Cluster for iso_label_2.0 with leiden


/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/isolated_labels.py:311: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  batch_per_lab = tmp.groupby(label_key).agg({batch_key: "count"})
  0%|          | 0/9 [00:00<?, ?it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/preprocessing/_pca/__init__.py:226: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can b

mean silhouette per group:                silhouette_score
group                          
B cells                0.822475
CAFs                   0.893751
Cholangiocyte          0.783569
Endothelial            0.848822
Erythroid              0.666667
Fibroblast             0.834107
Hepatocyte             0.883666
Kupffer cells          0.624767
LSECs                  0.881755
Malignant              0.963871
Mesenchymal            0.932752
Myeloid                0.909928
NK cells               0.965425
Plasma cells           0.874466
T cells                0.899947
TAMs                   0.969042
TECs                   0.892078


/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/preprocessing/_pca/__init__.py:226: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/preprocessing/_pca/__init__.py:226: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/kbet.py:111: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass 

2 labels consist of a single batch or is too small. Skip.
Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 6
Adding diffusion to step 7
Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 4
Adding diffusion to step 4
Adding diffusion to step 4
Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 6
Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 6
Adding diffusion to step 4
Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 6
Adding diffusion to step 7
Adding diffusion to step 4


# scvi

In [5]:
import importlib
import thesis_project.Integration.Integration_methods.scvi as scvi_mod
importlib.reload(scvi_mod)

from thesis_project.Integration.Integration_methods.scvi import run, ScviConfig

cfg = ScviConfig(
    batch_key="dataset",
    label_key="major_celltype_l1",  
    run_tag="scvi_smoke",
    save_h5ad=False,
)

outdir = PROJECT_ROOT / "Results" / "Smoke" / "scvi_smoke"

ad_out, metrics, perf = run(
    adata_in=ad_smoke,
    outdir=str(outdir),
    cfg=cfg,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


2026-04-07 13:00:36 | [scVI] accelerator=gpu
2026-04-07 13:00:36 | [scVI] covariates=['platform']
2026-04-07 13:00:36 | [scVI] setup_anndata: start
2026-04-07 13:00:36 | [scVI] setup_anndata: done
2026-04-07 13:00:36 | [scVI] init model: n_latent=50, n_hidden=128, n_layers=2, gene_likelihood=nb
2026-04-07 13:00:36 | [scVI] train: start


/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/lightning/pytorch/loops/fit_loop.py:317: The number of training batches (9) is smaller than the logging interval Trainer(log_every_n_steps=10). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training:   0%|          | 0/200 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=200` reached.


2026-04-07 13:02:46 | [scVI] train: done
2026-04-07 13:02:46 | [scVI] saving model...
2026-04-07 13:02:46 | [scVI] model saved
Cluster for iso_label_0.2 with leiden
Cluster for iso_label_0.4 with leiden
Cluster for iso_label_0.6 with leiden
Cluster for iso_label_0.8 with leiden
Cluster for iso_label_1.0 with leiden
Cluster for iso_label_1.2 with leiden
Cluster for iso_label_1.4 with leiden
Cluster for iso_label_1.6 with leiden
Cluster for iso_label_1.8 with leiden
Cluster for iso_label_2.0 with leiden


  0%|          | 0/9 [00:00<?, ?it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
 11%|█         | 1/9 [00:00<00:05,  1.51it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
 22%|██▏       | 2/9 [00:01<00:03,  1.95it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
 33%|███▎      | 3/9 [00:01<00:02,  2.71it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: Im

mean silhouette per group:                silhouette_score
group                          
B cells                0.803736
CAFs                   0.827199
Cholangiocyte          0.853725
Endothelial            0.892272
Erythroid              0.333333
Fibroblast             0.825574
Hepatocyte             0.712617
Kupffer cells          0.734350
LSECs                  0.588263
Malignant              0.863332
Mesenchymal            0.689772
Myeloid                0.875585
NK cells               0.880221
Plasma cells           0.778007
T cells                0.886789
TAMs                   0.943740
TECs                   0.929659
2 labels consist of a single batch or is too small. Skip.
Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 6
Adding diffusion to step 7
Adding diffusion to step 4
Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 4
Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 6
Adding dif

# BBKNN
subset → PCA → BBKNN neighbor graph (batch-balanced) → diffusion map → UMAP + Leiden → scIB-like metrics → plots

- For graph-based integration methods such as BBKNN, integration quality was evaluated on a diffusion map embedding derived from the batch-balanced kNN graph, rather than on uncorrected PCA coordinates. Canonical kBET was computed using the scIB implementation backed by the R package kBET. To ensure comparability across metrics, all scores were transformed to a “higher-is-better” convention after validating their numeric ranges and definitions.




In [12]:
import importlib
import thesis_project.Integration.Integration_benchmark.scib_compat as scib_compat
importlib.reload(scib_compat)
from thesis_project.Integration.Integration_methods import bbknn as bbknn
from thesis_project.Integration.Integration_methods.bbknn import run, BBKNNConfig #runner for BBKNN integration in Scanpy
importlib.reload(bbknn)

import thesis_project.Integration.Integration_benchmark.metrics as metrics_mod
importlib.reload(metrics_mod)


cfg = BBKNNConfig(
    batch_key="dataset",
    label_key="major_celltype_l1",
    run_tag="esraa_bbknn_smoke",
    save_h5ad=False, 
)

outdir = PROJECT_ROOT / "Results" / "Smoke" / "bbknn_smoke"
ad_out, metrics, perf = run(
    adata_in=ad_smoke,
    outdir=str(outdir),
    cfg=cfg,
) 


Cluster for iso_label_0.2 with leiden
Cluster for iso_label_0.4 with leiden
Cluster for iso_label_0.6 with leiden
Cluster for iso_label_0.8 with leiden
Cluster for iso_label_1.0 with leiden
Cluster for iso_label_1.2 with leiden
Cluster for iso_label_1.4 with leiden
Cluster for iso_label_1.6 with leiden
Cluster for iso_label_1.8 with leiden
Cluster for iso_label_2.0 with leiden


  0%|          | 0/9 [00:00<?, ?it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
 11%|█         | 1/9 [00:00<00:04,  1.77it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
 22%|██▏       | 2/9 [00:00<00:03,  2.10it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
 33%|███▎      | 3/9 [00:01<00:02,  2.86it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: Im

mean silhouette per group:                silhouette_score
group                          
B cells                0.699422
CAFs                   0.750025
Cholangiocyte          0.784356
Endothelial            0.784443
Erythroid              0.828994
Fibroblast             0.690913
Hepatocyte             0.849572
Kupffer cells          0.611059
LSECs                  0.768844
Malignant              0.909093
Mesenchymal            0.906378
Myeloid                0.857603
NK cells               0.842771
Plasma cells           0.829591
T cells                0.779106
TAMs                   0.922861
TECs                   0.821260
Adding diffusion to step 2
2 labels consist of a single batch or is too small. Skip.


# Harmony

In [7]:
from thesis_project.Integration.Integration_methods.harmony import run, HarmonyConfig

cfg = HarmonyConfig(
    batch_key="dataset",
    label_key="major_celltype_l1",
    run_tag="harmony_smoke",
    save_h5ad=False,
)

outdir = PROJECT_ROOT / "Results" / "Smoke" / "harmony_smoke"

ad_out, metrics, perf = run(
    adata_in=ad_smoke,
    outdir=str(outdir),
    cfg=cfg,
)


2026-04-07 13:12:01,020 - harmonypy - INFO - Running Harmony (PyTorch on cuda)
2026-04-07 13:12:01,021 - harmonypy - INFO -   Parameters:
2026-04-07 13:12:01,021 - harmonypy - INFO -     max_iter_harmony: 20
2026-04-07 13:12:01,021 - harmonypy - INFO -     max_iter_kmeans: 20
2026-04-07 13:12:01,021 - harmonypy - INFO -     epsilon_cluster: 1e-05
2026-04-07 13:12:01,022 - harmonypy - INFO -     epsilon_harmony: 0.0001
2026-04-07 13:12:01,022 - harmonypy - INFO -     nclust: 100
2026-04-07 13:12:01,022 - harmonypy - INFO -     block_size: 0.05
2026-04-07 13:12:01,023 - harmonypy - INFO -     lamb: [1. 1. 1. 1. 1. 1. 1. 1. 1.]
2026-04-07 13:12:01,023 - harmonypy - INFO -     theta: [2. 2. 2. 2. 2. 2. 2. 2. 2.]
2026-04-07 13:12:01,024 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-04-07 13:12:01,024 - harmonypy - INFO -     verbose: True
2026-04-07 13:12:01,024 - harmonypy - INFO -     random_state: 0
2026-04-07 13:12:01,024 - harmonypy - INFO -   Data: 50 PCs × 19997 cells

Cluster for iso_label_0.2 with leiden
Cluster for iso_label_0.4 with leiden
Cluster for iso_label_0.6 with leiden
Cluster for iso_label_0.8 with leiden
Cluster for iso_label_1.0 with leiden
Cluster for iso_label_1.2 with leiden
Cluster for iso_label_1.4 with leiden
Cluster for iso_label_1.6 with leiden
Cluster for iso_label_1.8 with leiden
Cluster for iso_label_2.0 with leiden


  0%|          | 0/9 [00:00<?, ?it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
 11%|█         | 1/9 [00:00<00:04,  1.72it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
 22%|██▏       | 2/9 [00:00<00:03,  2.06it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
 33%|███▎      | 3/9 [00:01<00:02,  2.84it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: Im

mean silhouette per group:                silhouette_score
group                          
B cells                0.812007
CAFs                   0.941859
Cholangiocyte          0.893403
Endothelial            0.916577
Erythroid              0.799279
Fibroblast             0.883533
Hepatocyte             0.828029
Kupffer cells          0.828440
LSECs                  0.885699
Malignant              0.839092
Mesenchymal            0.574651
Myeloid                0.931663
NK cells               0.941422
Plasma cells           0.826522
T cells                0.925286
TAMs                   0.953431
TECs                   0.946423
2 labels consist of a single batch or is too small. Skip.
Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 6
Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 6
Adding diffusion to step 7
Adding diffusion to step 8
Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 4
Adding dif

# scanorama

In [8]:
from thesis_project.Integration.Integration_methods.scanorama import run, ScanoramaConfig
import importlib
import thesis_project.Integration.Integration_methods.scanorama as scanorama
importlib.reload(scanorama)
cfg = ScanoramaConfig(
    batch_key="dataset",
    label_key="major_celltype_l1",
    run_tag="scanorama_smoke",
    save_h5ad=False,
)

outdir = PROJECT_ROOT / "Results" / "Smoke" / "scanorama_smoke"

ad_out, metrics, perf = run(
    adata_in=ad_smoke,
    outdir=str(outdir),
    cfg=cfg,
)


2026-04-07 13:18:29 | [HVG] Input already HVG-restricted (4000 genes).
2026-04-07 13:18:29 | [Scanorama] backend=scanorama | batches=9 | dimred=50, knn=30, alpha=0.1, sigma=20.0, approx=True, batch_size=5000
Found 4000 genes among all datasets
[[0.           0.6151596941 0.4185185185 0.4964028777 0.5531864487
  0.0838574423 0.4358316222 0.3060234814 0.025147929 ]
 [0.           0.           0.9814814815 0.4982014388 0.627339493
  0.4591194969 0.3916837782 0.6271056662 0.0473372781]
 [0.           0.           0.           0.0444444444 0.3537037037
  0.3249475891 0.3703703704 0.5203703704 0.0259259259]
 [0.           0.           0.           0.           0.2643884892
  0.2194244604 0.1816546763 0.3345323741 0.0629496403]
 [0.           0.           0.           0.           0.
  0.0545073375 0.386036961  0.7803837953 0.2337278107]
 [0.           0.           0.           0.           0.
  0.           0.035639413  0.3438155136 0.7463312369]
 [0.           0.           0.           0.  

  0%|          | 0/9 [00:00<?, ?it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
 11%|█         | 1/9 [00:00<00:05,  1.44it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
 22%|██▏       | 2/9 [00:01<00:03,  1.81it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
 33%|███▎      | 3/9 [00:01<00:02,  2.54it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: Im

mean silhouette per group:                silhouette_score
group                          
B cells                0.858731
CAFs                   0.916116
Cholangiocyte          0.827925
Endothelial            0.896101
Erythroid              0.631537
Fibroblast             0.837915
Hepatocyte             0.721169
Kupffer cells          0.681097
LSECs                  0.897311
Malignant              0.886090
Mesenchymal            0.649473
Myeloid                0.920871
NK cells               0.924080
Plasma cells           0.869625
T cells                0.939494
TAMs                   0.934949
TECs                   0.936019
2 labels consist of a single batch or is too small. Skip.
Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 4
Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 6
Adding diffusion to step 7
Adding diffusion to step 8
Adding diffusion to step 9
Adding diffusion to step 10
Adding diffusion to step 4
Adding di

# scanvi

In [13]:
from thesis_project.Integration.Integration_methods.scanvi import run, ScanviConfig

cfg = ScanviConfig(
    batch_key="dataset",
    label_key="major_celltype_l1",
    counts_layer="counts",
    run_tag="scanvi_smoke",
    save_h5ad=True,   
)

outdir = PROJECT_ROOT / "Results" / "Smoke" / "scanvi_smoke"

ad_out, metrics, perf = run(
    adata_in=ad_smoke,
    outdir=str(outdir),
    cfg=cfg,
)


INFO     Training for 200 epochs.                                                                                  


/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scvi/data/fields/_arraylike_field.py:421: UserWarning: Category 38 in adata.obs['donor_id'] has fewer than 3 cells. Models may not train properly.
  mapping = _make_column_categorical(df, key, key, categorical_dtype=categorical_dtype)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/lightning/pytorch/loops/fit_loop.py:317: The number of training batches (9) is smaller than the logging interval Trainer(log_every_n_steps=10). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training:   0%|          | 0/200 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=200` reached.


Recompute neighbors on rep X_scanvi_smoke instead of None
Cluster for iso_label_0.2 with leiden
Cluster for iso_label_0.4 with leiden
Cluster for iso_label_0.6 with leiden
Cluster for iso_label_0.8 with leiden
Cluster for iso_label_1.0 with leiden
Cluster for iso_label_1.2 with leiden
Cluster for iso_label_1.4 with leiden
Cluster for iso_label_1.6 with leiden
Cluster for iso_label_1.8 with leiden
Cluster for iso_label_2.0 with leiden


  0%|          | 0/9 [00:00<?, ?it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
 11%|█         | 1/9 [00:00<00:04,  1.71it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
 22%|██▏       | 2/9 [00:01<00:03,  2.06it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
 33%|███▎      | 3/9 [00:01<00:02,  2.63it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: Im

mean silhouette per group:                silhouette_score
group                          
B cells                0.723414
CAFs                   0.823758
Cholangiocyte          0.867068
Endothelial            0.830254
Erythroid              0.333333
Fibroblast             0.789675
Hepatocyte             0.739553
Kupffer cells          0.785703
LSECs                  0.526233
Malignant              0.864489
Mesenchymal            0.602782
Myeloid                0.856785
NK cells               0.802421
Plasma cells           0.769881
T cells                0.889859
TAMs                   0.909133
TECs                   0.885816
2 labels consist of a single batch or is too small. Skip.
Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 6
Adding diffusion to step 7
Adding diffusion to step 8
Adding diffusion to step 4
Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 4
Adding dif

# Combat

In [17]:
import importlib
from thesis_project.Integration.Integration_methods.combat import run, CombatConfig
importlib.reload(importlib.import_module("thesis_project.Integration.Integration_methods.combat"))

cfg = CombatConfig(
    combat_space="expr",
    input_layer=None,         
    require_hvg=True,
    max_hvgs=4000,
    plot_subsample_n=100000,
)

outdir = PROJECT_ROOT / "Results" / "Smoke" / "combat_smoke"

ad_out, metrics, perf = run(
    adata_in=ad_smoke,
    outdir=str(outdir),
    cfg=cfg,
) 

/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/preprocessing/_combat.py:344: RuntimeWarning: divide by zero encountered in divide
  (abs(g_new - g_old) / g_old).max(), (abs(d_new - d_old) / d_old).max()


Recompute neighbors on rep X_pca_combat instead of None
Cluster for iso_label_0.2 with leiden
Cluster for iso_label_0.4 with leiden
Cluster for iso_label_0.6 with leiden
Cluster for iso_label_0.8 with leiden
Cluster for iso_label_1.0 with leiden
Cluster for iso_label_1.2 with leiden
Cluster for iso_label_1.4 with leiden
Cluster for iso_label_1.6 with leiden
Cluster for iso_label_1.8 with leiden
Cluster for iso_label_2.0 with leiden


  0%|          | 0/9 [00:00<?, ?it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
 11%|█         | 1/9 [00:00<00:04,  1.64it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
 22%|██▏       | 2/9 [00:01<00:03,  1.98it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[score_name] = pd.Series(
 33%|███▎      | 3/9 [00:01<00:02,  2.63it/s]/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scanpy/tools/_score_genes.py:165: Im

mean silhouette per group:                silhouette_score
group                          
B cells                0.714509
CAFs                   0.843133
Cholangiocyte          0.801328
Endothelial            0.836744
Erythroid              0.732048
Fibroblast             0.821849
Hepatocyte             0.727246
Kupffer cells          0.784309
LSECs                  0.804108
Malignant              0.812741
Mesenchymal            0.782275
Myeloid                0.835875
NK cells               0.842907
Plasma cells           0.735152
T cells                0.794891
TAMs                   0.859621
TECs                   0.803028
2 labels consist of a single batch or is too small. Skip.
Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 4
Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 6
Adding diffusion to step 4
Adding diffusion to step 4
Adding diffusion to step 5
Adding dif

TypeError: plot_marker_dotplot_pub() got an unexpected keyword argument 'adata'

# Seurat

In [3]:
from pathlib import Path
import importlib
from thesis_project.Integration.Integration_methods import seurat as seurat_mod
importlib.reload(seurat_mod)

PROJECT_ROOT = Path("/data1/esraa/Thesis-Project")
outdir = PROJECT_ROOT / "Results" / "Smoke" / "seurat_rpca_test"
outdir.mkdir(parents=True, exist_ok=True)

cfg = seurat_mod.SeuratConfig(
    batch_key="dataset",
    label_key="major_celltype_l1",
    run_tag="seurat_smoke_rpca",
    mode="rpca",
    dims=20,
    n_pcs=50,
    require_hvg=True,  # Use HVG subset (data already filtered to HVGs)
    max_hvgs=4000,
    # If your raw counts are in a layer, set this:
    # input_layer_counts="counts",
)

ad_out, metrics, perf = seurat_mod.run_seurat_via_r(ad_smoke, str(outdir), cfg)

[Python] Exporting ad.X as-is for R.
[Python] Writing temporary h5ad for R: /data1/esraa/Thesis-Project/Results/Smoke/seurat_rpca_test/adata_for_seurat_seurat_smoke_rpca.h5ad


[Python] Wrote HVG list: /data1/esraa/Thesis-Project/Results/Smoke/seurat_rpca_test/seurat_genes_seurat_smoke_rpca.csv (n=4000)
[Python] Calling R Seurat with: Rscript /data1/esraa/Thesis-Project/src/thesis_project/Integration/Integration_methods/R/run_seurat_integration.R /data1/esraa/Thesis-Project/Results/Smoke/seurat_rpca_test/adata_for_seurat_seurat_smoke_rpca.h5ad dataset /data1/esraa/Thesis-Project/Results/Smoke/seurat_rpca_test/seurat_smoke_rpca rpca 20 20 50 /data1/esraa/Thesis-Project/Results/Smoke/seurat_rpca_test/seurat_genes_seurat_smoke_rpca.csv 0
[R] Reading H5AD: /data1/esraa/Thesis-Project/Results/Smoke/seurat_rpca_test/adata_for_seurat_seurat_smoke_rpca.h5ad
[R] Warning message:
[R] The names of these selected obs columns have been modified to match R
[R] conventions: 'Cell#' -> 'Cell.' and 'Cluster#' -> 'Cluster.'
[R] Batches found: GSE115469, GSE124395, GSE125449, GSE136103, GSE138709, GSE140228, GSE146409, GSE149614, GSE151530
[R] Assays in SCE: X, counts, log1p_no

/data1/esraa/Thesis-Project/src/thesis_project/Integration/Integration_benchmark/graph.py:232: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/isolated_labels.py:311: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  batch_per_lab = tmp.groupby(label_key).agg({batch_key: "count"})


Recompute neighbors on rep X_emb instead of None
Cluster for iso_label_0.2 with leiden


/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/clustering.py:96: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  cluster_function(adata, resolution=res, key_added=resolution_key, **kwargs)


...

/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Se

Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 6


/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


Adding diffusion to step 4
Adding diffusion to step 5
Adding diffusion to step 6
Adding diffusion to step 7


/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/kbet.py:158: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  comp_size = pd.value_counts(labs)
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/kbet.py:229: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/graph_connectivity.py:56: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(labels)
/data1/esraa/miniconda3/envs/thesis-env/lib/python3.10/site-packages/scib/metrics/graph_connectivity.py:56: FutureWarning: pandas.value_counts is deprecated and will be removed in a future ver